# The Greeks Bearing Nets — Full Pipeline (Colab)

**Run this notebook on Google Colab with a T4 GPU runtime.**

Workflow:
1. Mount Google Drive for persistence
2. Clone repo + install deps
3. Upload manual data files (CBOE, Kaggle)
4. Run pipeline section by section — checkpoints save to Drive so you survive disconnects

**Runtime setup:** Runtime > Change runtime type > T4 GPU

---
## 0. Environment Setup

Run this section every time the runtime (re)starts.

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Project lives on Drive so it survives disconnects
import os
PROJECT_DIR = '/content/drive/MyDrive/greeks-bearing-nets'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project dir: {PROJECT_DIR}')

In [ ]:
# Clone repo (first time) or pull updates
REPO_URL = 'https://github.com/dma2459/greeks-bearing-nets.git'

REPO_DIR = '/content/greeks-bearing-nets'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Install dependencies
!pip install -q yfinance fredapi tqdm scikit-learn

In [ ]:
# Symlink data and checkpoints to Drive (persist across disconnects)
import os

drive_dirs = {
    'data/raw':       f'{PROJECT_DIR}/data/raw',
    'data/processed': f'{PROJECT_DIR}/data/processed',
    'checkpoints':    f'{PROJECT_DIR}/checkpoints',
    'results':        f'{PROJECT_DIR}/results',
}

for local, remote in drive_dirs.items():
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local):
        os.unlink(local)
    elif os.path.exists(local):
        # Move any existing contents to Drive first
        !cp -rn {local}/* {remote}/ 2>/dev/null; rm -rf {local}
    os.symlink(remote, local)
    print(f'  {local} -> {remote}')

print('\nDrive symlinks ready. Data persists across runtime restarts.')

In [ ]:
# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    device = torch.device('cuda')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')
    device = torch.device('cpu')

# Add repo to path
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

---
## 1. Data Download

Automated sources download here. Manual files must be uploaded to `data/raw/` on Drive.

In [ ]:
# Check if already downloaded
spy_path = os.path.join('data', 'raw', 'spy.csv')
if os.path.exists(spy_path):
    print('Market data already exists on Drive. Skipping download.')
    print('Delete data/raw/spy.csv to force re-download.')
else:
    # Downloads: yfinance (SPY,VIX,VVIX,HYG,LQD,DXY,VIX9D,VIX3M,VIX6M),\n",
    #            CBOE put/call ratio, FRED treasury yields\n",
    download_all(
        fred_api_key=None,  # Set your FRED key here, or set FRED_API_KEY env var
    )


In [ ]:
# FRED treasury yields are downloaded by download_all() above.
# If you need a FRED API key, set it in the download_all() call.
# As a fallback, you can also manually place treasury.csv in data/raw/.
treasury_path = os.path.join('data', 'raw', 'treasury.csv')
print(f'Treasury data exists: {os.path.exists(treasury_path)}')


In [ ]:
# The ONLY manual download needed: Kaggle SPY options data
# Everything else is automated now.
raw_dir = os.path.join('data', 'raw')

# Check for options file (the only manual download)
opts_found = any(
    os.path.exists(os.path.join(raw_dir, f))
    for f in ['spy_options.csv', 'spy_options_data.csv', 'sp500_options.csv',
              'options.csv', 'spy_options_eod.csv', 'cleaned_options_data.csv']
)

if not opts_found:
    # Check subdirectories too (Kaggle zips extract into folders)
    for item in os.listdir(raw_dir) if os.path.exists(raw_dir) else []:
        subpath = os.path.join(raw_dir, item)
        if os.path.isdir(subpath):
            for f in os.listdir(subpath):
                if f.endswith('.csv'):
                    opts_found = True
                    break

if opts_found:
    print('Options data found.')
else:
    print('OPTIONS DATA NOT FOUND.')
    print()
    print('Download from Kaggle and upload to Google Drive:')
    print('  https://www.kaggle.com/datasets/benjaminbtang/spy-options-2010-2023-eod')
    print(f'  Place CSV in: {PROJECT_DIR}/data/raw/')
    print()
    print('Or upload directly here (uncomment next lines):')
    # from google.colab import files
    # uploaded = files.upload()
    # for name, data in uploaded.items():
    #     with open(os.path.join(raw_dir, name), 'wb') as f:
    #         f.write(data)


In [ ]:
# Verify all data
import glob
files = sorted(glob.glob(os.path.join(raw_dir, '*.csv')))
print(f'Files in data/raw/ ({len(files)}):')
for f in files:
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f):30s}  {size_mb:>8.2f} MB')

---
## 2. Preprocessing

In [ ]:
import numpy as np
import pandas as pd

from src.data.download import load_cboe_files
from src.data.preprocess import (
    build_master_dataframe, clean_and_split, fit_scaler,
    scale_data, build_sequences, preprocess_options, split_options,
    FEATURE_COLS, SEQ_LEN
)

RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

# Check if already preprocessed
if os.path.exists(os.path.join(PROCESSED_DIR, 'train_sequences.npy')):
    print('Preprocessed data found on Drive. Loading...')
    master = pd.read_csv(os.path.join(PROCESSED_DIR, 'master_df.csv'), index_col=0, parse_dates=True)
    train_seqs = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))
    print(f'Master: {master.shape}, Train sequences: {train_seqs.shape}')
    SKIP_PREPROCESS = True
else:
    SKIP_PREPROCESS = False
    print('No preprocessed data found. Running preprocessing...')

In [ ]:
if not SKIP_PREPROCESS:
    # Load raw data
    spy = pd.read_csv(os.path.join(RAW_DIR, 'spy.csv'), index_col=0, parse_dates=True)
    vix = pd.read_csv(os.path.join(RAW_DIR, 'vix.csv'), index_col=0, parse_dates=True)
    
    vvix_path = os.path.join(RAW_DIR, 'vvix.csv')
    vvix = pd.read_csv(vvix_path, index_col=0, parse_dates=True) if os.path.exists(vvix_path) else None
    
    hyg = pd.read_csv(os.path.join(RAW_DIR, 'hyg.csv'), index_col=0, parse_dates=True)
    lqd = pd.read_csv(os.path.join(RAW_DIR, 'lqd.csv'), index_col=0, parse_dates=True)
    dxy = pd.read_csv(os.path.join(RAW_DIR, 'dxy.csv'), index_col=0, parse_dates=True)
    
    treasury_path = os.path.join(RAW_DIR, 'treasury.csv')
    treasury = pd.read_csv(treasury_path, index_col=0, parse_dates=True) if os.path.exists(treasury_path) else None
    
    cboe = load_cboe_files(RAW_DIR)
    
    # Build master DataFrame
    master = build_master_dataframe(spy, vix, vvix, hyg, lqd, dxy, treasury, cboe)
    print(f'Master: {master.shape}, Date range: {master.index[0].date()} to {master.index[-1].date()}')
    
    # Clean, split, scale
    train_df, test_df = clean_and_split(master)
    print(f'Train: {len(train_df)} days, Test: {len(test_df)} days')
    
    scaler = fit_scaler(train_df, os.path.join(PROCESSED_DIR, 'scaler.pkl'))
    train_scaled = scale_data(train_df, scaler)
    test_scaled = scale_data(test_df, scaler)
    
    # Build sequences
    train_seqs = build_sequences(train_scaled, seq_len=SEQ_LEN)
    print(f'Training sequences: {train_seqs.shape}')
    
    # Save everything
    master.to_csv(os.path.join(PROCESSED_DIR, 'master_df.csv'))
    np.save(os.path.join(PROCESSED_DIR, 'train_sequences.npy'), train_seqs)
    np.save(os.path.join(PROCESSED_DIR, 'train_scaled.npy'), train_scaled)
    np.save(os.path.join(PROCESSED_DIR, 'test_scaled.npy'), test_scaled)
    train_df.index.to_frame().to_csv(os.path.join(PROCESSED_DIR, 'train_dates.csv'))
    test_df.index.to_frame().to_csv(os.path.join(PROCESSED_DIR, 'test_dates.csv'))
    print('Saved to Drive.')

In [ ]:
# Preprocess options (if file exists)
import pickle

opts_test_path = os.path.join(PROCESSED_DIR, 'opts_test.pkl')
if os.path.exists(opts_test_path):
    print('Options data already preprocessed.')
else:
    opts_file = None
    for fname in ['spy_options.csv', 'spy_options_data.csv', 'sp500_options.csv', 'options.csv']:
        p = os.path.join(RAW_DIR, fname)
        if os.path.exists(p):
            opts_file = p
            break
    
    if opts_file:
        with open(os.path.join(PROCESSED_DIR, 'scaler.pkl'), 'rb') as f:
            scaler = pickle.load(f)
        
        opts_raw = pd.read_csv(opts_file, parse_dates=['date', 'expiration'])
        print(f'Raw options: {len(opts_raw)} rows')
        
        opts = preprocess_options(opts_raw, master, scaler, seq_len=SEQ_LEN)
        opts_train, opts_test = split_options(opts)
        print(f'Train: {len(opts_train)}, Test: {len(opts_test)}')
        
        opts_train.to_pickle(os.path.join(PROCESSED_DIR, 'opts_train.pkl'))
        opts_test.to_pickle(os.path.join(PROCESSED_DIR, 'opts_test.pkl'))
        print('Saved to Drive.')
    else:
        print('No options CSV found. Upload spy_options.csv to data/raw/ first.')

---
## 3. EDA (Quick Sanity Checks)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

master = pd.read_csv(os.path.join(PROCESSED_DIR, 'master_df.csv'), index_col=0, parse_dates=True)
FIGURES_DIR = 'results/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# SPY price
master['spy_close'].plot(ax=axes[0,0], title='SPY Close')
axes[0,0].axvline(pd.Timestamp('2020-01-01'), color='red', ls='--', label='Train/Test')
axes[0,0].legend()

# Volatility
for col in ['rv_21d', 'vix', 'gk_vol']:
    if col in master.columns:
        master[col].plot(ax=axes[0,1], alpha=0.7, label=col)
axes[0,1].set_title('Volatility Features')
axes[0,1].legend()

# Return distribution
returns = master['log_return'].dropna()
axes[1,0].hist(returns, bins=100, density=True, alpha=0.7)
axes[1,0].set_title(f'Log Returns (kurtosis={returns.kurtosis():.1f})')

# Regime distribution
regime_names = {0: 'Low', 1: 'Med', 2: 'High'}
for period, mask in [('Train', master.index < '2020'), ('Test', master.index >= '2020')]:
    counts = master.loc[mask, 'vol_regime'].value_counts().sort_index()
    axes[1,1].bar([f'{regime_names.get(int(k), k)}\n{period}' for k in counts.index],
                   counts.values, alpha=0.7, label=period)
axes[1,1].set_title('Vol Regime Counts')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_overview.png'), dpi=150, bbox_inches='tight')
plt.show()

# Correlation
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(master[FEATURE_COLS].corr(), annot=True, fmt='.1f', cmap='RdBu_r',
            center=0, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlations')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 4. TimeGAN Training

~30-60 min on T4 GPU. Checkpoints save to Drive every 100 epochs.

In [ ]:
from src.models.timegan import TimeGAN
from src.training.train_gan import train_timegan, generate_synthetic_sequences

train_seqs = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))
GAN_CHECKPOINT = 'checkpoints/timegan'

# Check if already trained
gan_final_path = os.path.join(GAN_CHECKPOINT, 'timegan_final.pt')
if os.path.exists(gan_final_path):
    print('Trained GAN found on Drive. Loading...')
    model_gan = TimeGAN(input_dim=20, hidden_dim=64)
    model_gan.load_state_dict(torch.load(gan_final_path, map_location='cpu'))
    print('Loaded.')
    SKIP_GAN_TRAIN = True
else:
    SKIP_GAN_TRAIN = False
    print(f'No trained GAN found. Will train from scratch.')
    print(f'Training sequences: {train_seqs.shape}')

In [ ]:
if not SKIP_GAN_TRAIN:
    model_gan, history_gan = train_timegan(
        train_seqs,
        input_dim=20,
        hidden_dim=64,
        batch_size=128,
        phase1_epochs=200,
        phase2_epochs=200,
        phase3_epochs=500,
        lr=1e-3,
        device=device,
        checkpoint_dir=GAN_CHECKPOINT,
        log_every=20,
    )
    
    # Plot training curves
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history_gan['phase1_loss']); axes[0].set_title('Phase 1: AE')
    axes[1].plot(history_gan['phase2_loss']); axes[1].set_title('Phase 2: Supervisor')
    axes[2].plot(history_gan['phase3_d_loss'], label='D')
    axes[2].plot(history_gan['phase3_g_loss'], label='G')
    axes[2].set_title('Phase 3: Adversarial'); axes[2].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'gan_training_curves.png'), dpi=150)
    plt.show()

In [ ]:
# Generate 50K synthetic sequences
gan_seqs_path = os.path.join(PROCESSED_DIR, 'gan_sequences.npy')
if os.path.exists(gan_seqs_path):
    print('Synthetic sequences already on Drive.')
    gan_seqs = np.load(gan_seqs_path)
else:
    gan_seqs = generate_synthetic_sequences(
        model_gan, n_samples=50000, seq_len=60, batch_size=1000,
        device=device, save_path=gan_seqs_path,
    )
print(f'Synthetic sequences: {gan_seqs.shape}')

---
## 5. GAN Quality Checks

Must pass before proceeding.

In [ ]:
from src.evaluation.gan_eval import run_all_quality_checks

train_seqs = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))
gan_seqs = np.load(os.path.join(PROCESSED_DIR, 'gan_sequences.npy'))

n_val = len(train_seqs) // 5
results_gan = run_all_quality_checks(
    train_seqs[:-n_val], train_seqs[-n_val:], gan_seqs,
    figures_dir=FIGURES_DIR,
)

print(f'\nKurtosis — Real: {results_gan["real_kurtosis"]:.2f}, Synthetic: {results_gan["fake_kurtosis"]:.2f}')
print(f'MMD — Baseline: {results_gan["baseline_mmd"]:.6f}, GAN: {results_gan["gan_mmd"]:.6f}')
print(f'MMD check: {"PASS" if results_gan["passed"] else "FAIL"}')

---
## 6. Generate Transformer Training Data

50K Heston + 50K GAN labeled samples.

In [ ]:
from src.models.heston import simulate_heston
from tqdm import tqdm

heston_labels_path = os.path.join(PROCESSED_DIR, 'heston_labels.npy')

if os.path.exists(heston_labels_path):
    print('Heston data already on Drive.')
    heston_contracts = np.load(os.path.join(PROCESSED_DIR, 'heston_contracts.npy'))
    heston_labels = np.load(heston_labels_path)
else:
    N_HESTON = 50000
    rng = np.random.RandomState(42)

    S0_samples = rng.uniform(250, 500, N_HESTON)
    moneyness_samples = rng.uniform(0.85, 1.15, N_HESTON)
    K_samples = S0_samples / moneyness_samples
    T_samples = rng.uniform(7/365, 90/365, N_HESTON)
    r_samples = rng.uniform(0.001, 0.055, N_HESTON)

    heston_labels = np.zeros(N_HESTON)
    heston_contracts = np.zeros((N_HESTON, 4))

    for i in tqdm(range(N_HESTON), desc='Heston MC'):
        price = simulate_heston(
            S0=S0_samples[i], K=K_samples[i], T=T_samples[i], r=r_samples[i],
            n_paths=10000, n_steps=max(1, int(T_samples[i] * 252)), seed=42+i,
        )
        heston_labels[i] = price
        heston_contracts[i] = [K_samples[i], T_samples[i], r_samples[i], moneyness_samples[i]]

    np.save(os.path.join(PROCESSED_DIR, 'heston_contracts.npy'), heston_contracts)
    np.save(heston_labels_path, heston_labels)
    print(f'Saved. Mean label: ${heston_labels.mean():.2f}')

In [ ]:
# GAN-based training data
gan_labels_path = os.path.join(PROCESSED_DIR, 'gan_labels.npy')

if os.path.exists(gan_labels_path):
    print('GAN training data already on Drive.')
    gan_contracts = np.load(os.path.join(PROCESSED_DIR, 'gan_contracts.npy'))
    gan_labels = np.load(gan_labels_path)
else:
    gan_seqs = np.load(os.path.join(PROCESSED_DIR, 'gan_sequences.npy'))
    with open(os.path.join(PROCESSED_DIR, 'scaler.pkl'), 'rb') as f:
        scaler = pickle.load(f)
    
    log_return_mean = scaler.mean_[0]
    log_return_std = scaler.scale_[0]

    N_GAN = 50000
    rng_gan = np.random.RandomState(123)
    S0_gan = rng_gan.uniform(250, 500, N_GAN)
    moneyness_gan = rng_gan.uniform(0.85, 1.15, N_GAN)
    K_gan = S0_gan / moneyness_gan
    T_gan = rng_gan.uniform(7/365, 90/365, N_GAN)
    r_gan = rng_gan.uniform(0.001, 0.055, N_GAN)

    gan_labels = np.zeros(N_GAN)
    gan_contracts = np.zeros((N_GAN, 4))
    n_paths_per = 100

    for i in tqdm(range(N_GAN), desc='GAN pricing'):
        idx = rng_gan.choice(len(gan_seqs), n_paths_per, replace=False)
        batch = gan_seqs[idx]
        lr_scaled = batch[:, :, 0]
        lr_raw = lr_scaled * log_return_std + log_return_mean
        n_steps = min(max(1, int(T_gan[i] * 252)), 60)
        S_T = S0_gan[i] * np.exp(np.cumsum(lr_raw[:, :n_steps], axis=1)[:, -1])
        gan_labels[i] = np.exp(-r_gan[i] * T_gan[i]) * np.mean(np.maximum(S_T - K_gan[i], 0))
        gan_contracts[i] = [K_gan[i], T_gan[i], r_gan[i], moneyness_gan[i]]

    np.save(os.path.join(PROCESSED_DIR, 'gan_contracts.npy'), gan_contracts)
    np.save(gan_labels_path, gan_labels)
    print(f'Saved. Mean label: ${gan_labels.mean():.2f}')

---
## 7. Train Transformer (Experiments A, B, C)

~10-20 min each on T4.

In [ ]:
from src.models.transformer import build_transformer
from src.training.train_transformer import train_transformer
from src.data.dataset import SimulatedPricingDataset

train_seqs = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))
h_contracts = np.load(os.path.join(PROCESSED_DIR, 'heston_contracts.npy'))
h_labels = np.load(os.path.join(PROCESSED_DIR, 'heston_labels.npy'))
g_contracts = np.load(os.path.join(PROCESSED_DIR, 'gan_contracts.npy'))
g_labels = np.load(os.path.join(PROCESSED_DIR, 'gan_labels.npy'))

COMMON = dict(batch_size=256, max_epochs=100, lr=1e-4, patience=10, lr_patience=5, lr_factor=0.5, device=device)

histories = {}

In [ ]:
# Experiment A: Heston only
ckpt_a = 'checkpoints/transformer/experiment_a'
if os.path.exists(os.path.join(ckpt_a, 'best_model.pt')):
    print('Experiment A already trained.')
else:
    ds_a = SimulatedPricingDataset(train_seqs, h_contracts, h_labels)
    model_a, histories['A'] = train_transformer(
        build_transformer(), ds_a, checkpoint_dir=ckpt_a,
        experiment_name='Experiment A (Heston)', **COMMON)

In [ ]:
# Experiment B: GAN only
ckpt_b = 'checkpoints/transformer/experiment_b'
if os.path.exists(os.path.join(ckpt_b, 'best_model.pt')):
    print('Experiment B already trained.')
else:
    ds_b = SimulatedPricingDataset(train_seqs, g_contracts, g_labels)
    model_b, histories['B'] = train_transformer(
        build_transformer(), ds_b, checkpoint_dir=ckpt_b,
        experiment_name='Experiment B (GAN)', **COMMON)

In [ ]:
# Experiment C: Mixed
ckpt_c = 'checkpoints/transformer/experiment_c'
if os.path.exists(os.path.join(ckpt_c, 'best_model.pt')):
    print('Experiment C already trained.')
else:
    mixed_c = np.concatenate([h_contracts[:25000], g_contracts[:25000]])
    mixed_l = np.concatenate([h_labels[:25000], g_labels[:25000]])
    ds_c = SimulatedPricingDataset(train_seqs, mixed_c, mixed_l)
    model_c, histories['C'] = train_transformer(
        build_transformer(), ds_c, checkpoint_dir=ckpt_c,
        experiment_name='Experiment C (Mixed)', **COMMON)

In [ ]:
# Training curves
if histories:
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = {'A': 'blue', 'B': 'green', 'C': 'red'}
    for name, h in histories.items():
        ax.plot(h['val_loss'], color=colors[name], label=f'{name} (val)')
    ax.set_title('Transformer Validation Loss'); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
    ax.legend(); ax.set_yscale('log')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'transformer_training.png'), dpi=150)
    plt.show()

---
## 8. Evaluation on Real Options (2020-2023)

In [ ]:
from src.evaluation.pricing_eval import evaluate_all_models

opts_test = pd.read_pickle(os.path.join(PROCESSED_DIR, 'opts_test.pkl'))
master_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'master_df.csv'), index_col=0, parse_dates=True)
print(f'Test contracts: {len(opts_test)}')

# Load trained models
models = {}
for name, d in [('Transformer A (Heston)', 'experiment_a'),
                ('Transformer B (GAN)', 'experiment_b'),
                ('Transformer C (Mixed)', 'experiment_c')]:
    p = os.path.join('checkpoints/transformer', d, 'best_model.pt')
    if os.path.exists(p):
        m = build_transformer()
        m.load_state_dict(torch.load(p, map_location='cpu'))
        m.eval()
        models[name] = m
        print(f'  Loaded {name}')

models['Black-Scholes'] = 'BS'

In [ ]:
results = evaluate_all_models(
    opts_test, models, master_df=master_df, device=device,
    figures_dir=FIGURES_DIR, tables_dir='results/tables',
)

print('\n' + '=' * 60)
print('FINAL RESULTS')
print('=' * 60)
print(results.round(4).to_string())

---
## 9. Ablation Studies

In [ ]:
from src.models.transformer import build_ablation_transformer
from src.evaluation.pricing_eval import predict_transformer, compute_metrics

y_true = opts_test['mid_price'].values

# Baseline: Experiment C
baseline = build_transformer()
baseline.load_state_dict(torch.load('checkpoints/transformer/experiment_c/best_model.pt', map_location='cpu'))
baseline.eval()
baseline_preds = predict_transformer(baseline, opts_test, device=device)
baseline_mae = compute_metrics(y_true, baseline_preds)['MAE']
print(f'Baseline MAE: {baseline_mae:.4f}')

In [ ]:
train_seqs = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))
mixed_contracts = np.concatenate([h_contracts[:25000], g_contracts[:25000]])
mixed_labels = np.concatenate([h_labels[:25000], g_labels[:25000]])

def make_abl_seqs(abl_id, seqs):
    """Modify sequences for ablation."""
    if abl_id == 'A1': return np.delete(seqs, [6, 7], axis=2)
    if abl_id == 'A2': return np.delete(seqs, [14, 16, 17, 18], axis=2)
    if abl_id == 'A3': return seqs[:, -20:, :]
    if abl_id == 'A4':
        ts = np.load(os.path.join(PROCESSED_DIR, 'train_scaled.npy'))
        return np.array([ts[i:i+120] for i in range(len(ts)-120)], dtype=np.float32)
    if abl_id == 'A6': return np.delete(seqs, [19], axis=2)
    return seqs

ablation_models = {}
for abl_id in ['A1', 'A2', 'A3', 'A4', 'A5', 'A6']:
    ckpt = f'checkpoints/transformer/ablation_{abl_id.lower()}'
    if os.path.exists(os.path.join(ckpt, 'best_model.pt')):
        print(f'{abl_id}: already trained')
        seq_len = {'A3': 20, 'A4': 120}.get(abl_id, 60)
        m = build_ablation_transformer(abl_id, seq_len=seq_len)
        m.load_state_dict(torch.load(os.path.join(ckpt, 'best_model.pt'), map_location='cpu'))
        m.eval()
        ablation_models[abl_id] = m
    else:
        print(f'{abl_id}: training...')
        abl_seqs = make_abl_seqs(abl_id, train_seqs)
        seq_len = abl_seqs.shape[1]
        ds = SimulatedPricingDataset(abl_seqs, mixed_contracts, mixed_labels)
        m = build_ablation_transformer(abl_id, seq_len=seq_len)
        m, _ = train_transformer(m, ds, checkpoint_dir=ckpt,
                                 experiment_name=f'Ablation {abl_id}', **COMMON)
        ablation_models[abl_id] = m

In [ ]:
# Evaluate ablations
def predict_abl(model, opts_df, abl_id, device):
    """Predict with modified inputs for feature/seq-length ablations."""
    if abl_id == 'A5':  # Same input format, just fewer layers
        return predict_transformer(model, opts_df, device=device)
    
    model = model.to(device).eval()
    preds = []
    for _, row in opts_df.iterrows():
        seq = np.array(row['sequence'], dtype=np.float32)
        c = np.array([row['strike'], row['time_to_expiry'], row['rate_input'], row['moneyness_input']], dtype=np.float32)
        
        if abl_id == 'A1': seq = np.delete(seq, [6, 7], axis=1)
        elif abl_id == 'A2': seq = np.delete(seq, [14, 16, 17, 18], axis=1)
        elif abl_id == 'A3': seq = seq[-20:]
        elif abl_id == 'A4': seq = np.concatenate([np.zeros((60, seq.shape[1])), seq], axis=0)
        elif abl_id == 'A6': seq = np.delete(seq, [19], axis=1)
        
        ct = np.tile(c, (seq.shape[0], 1))
        inp = torch.tensor(np.concatenate([seq, ct], axis=-1), dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            preds.append(model(inp).cpu().item())
    return np.array(preds)

abl_rows = [{'Ablation': 'Baseline (C)', 'MAE': baseline_mae, 'Delta': 0.0}]
for abl_id, model in ablation_models.items():
    print(f'Evaluating {abl_id}...')
    preds = predict_abl(model, opts_test, abl_id, device)
    m = compute_metrics(y_true, preds)
    abl_rows.append({'Ablation': abl_id, 'MAE': m['MAE'], 'MSE': m['MSE'],
                     'MAPE': m['MAPE'], 'Delta': m['MAE'] - baseline_mae})
    print(f'  MAE: {m["MAE"]:.4f} (delta: {m["MAE"]-baseline_mae:+.4f})')

abl_df = pd.DataFrame(abl_rows)
abl_df.to_csv('results/tables/ablation_results.csv', index=False)
print('\n' + abl_df.to_string(index=False))

---
## Summary

In [ ]:
print('Pipeline complete. All results saved to Google Drive at:')
print(f'  {PROJECT_DIR}/results/tables/')
print(f'  {PROJECT_DIR}/results/figures/')
print(f'  {PROJECT_DIR}/checkpoints/')
print()
print('Files for the report:')
for d in ['results/figures', 'results/tables']:
    if os.path.exists(d):
        for f in sorted(os.listdir(d)):
            print(f'  {d}/{f}')